In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "323_build_gold_fact_project_item_comparison.py"
RUN_ID = str(uuid.uuid4())

FACT_PROJECT_ITEM_ESTIMATE_TABLE = f"{catalog}.gold.fact_project_item_estimate"
FACT_PROJECT_ITEM_BID_TABLE = f"{catalog}.gold.fact_project_item_bid"
FACT_PROJECT_BID_ROLE_CONTEXT_TABLE = f"{catalog}.gold.fact_project_bid_role_context"
TARGET_TABLE = f"{catalog}.gold.fact_project_item_comparison"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.fact_project_item_comparison
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    AS
    WITH estimate_base AS (
        SELECT
              e.*
            , ROW_NUMBER() OVER (
                PARTITION BY
                      e.project_id
                    , e.bid_item_sequence_number
                    , e.item_specification_key
                ORDER BY
                      e.estimate_item_rank ASC NULLS LAST
                    , e.estimate_item_fact_key DESC
              ) AS rn
        FROM {FACT_PROJECT_ITEM_ESTIMATE_TABLE} e
        WHERE e.is_standard_comparison_item = TRUE
    )

    , estimate_side AS (
        SELECT *
        FROM estimate_base
        WHERE rn = 1
    )

    , final_bid_base AS (
        SELECT
              b.*
            , ROW_NUMBER() OVER (
                PARTITION BY
                      b.project_id
                    , b.vendor_name_standardized
                    , b.bid_item_sequence_number
                    , b.item_specification_key
                ORDER BY
                      b.bid_submit_sequence_number DESC NULLS LAST
                    , b.bid_item_fact_key DESC
              ) AS rn
        FROM {FACT_PROJECT_ITEM_BID_TABLE} b
        WHERE b.is_latest_submit_for_vendor_project = TRUE
          AND COALESCE(b.item_missing_from_latest_project_submit, FALSE) = FALSE
    )

    , final_bid_items AS (
        SELECT *
        FROM final_bid_base
        WHERE rn = 1
    )

    , role_context AS (
        SELECT *
        FROM {FACT_PROJECT_BID_ROLE_CONTEXT_TABLE}
    )

    , focus_items AS (
        SELECT b.*
        FROM final_bid_items b
        JOIN role_context rc
          ON b.project_id = rc.project_id
         AND b.vendor_name_standardized = rc.focus_vendor_name_standardized
    )

    , benchmark_items AS (
        SELECT b.*
        FROM final_bid_items b
        JOIN role_context rc
          ON b.project_id = rc.project_id
         AND b.vendor_name_standardized = rc.benchmark_vendor_name_standardized
    )

    , item_universe AS (
        SELECT DISTINCT
              project_id
            , bid_item_sequence_number
            , item_specification_key
        FROM estimate_side

        UNION

        SELECT DISTINCT
              project_id
            , bid_item_sequence_number
            , item_specification_key
        FROM focus_items

        UNION

        SELECT DISTINCT
              project_id
            , bid_item_sequence_number
            , item_specification_key
        FROM benchmark_items
    )

    SELECT
          md5(
            concat_ws(
                '|',
                COALESCE(u.project_id, ''),
                COALESCE(CAST(u.bid_item_sequence_number AS STRING), ''),
                COALESCE(CAST(u.item_specification_key AS STRING), '')
            )
          ) AS project_item_comparison_key

        , u.project_id
        , u.bid_item_sequence_number
        , u.item_specification_key

        , fi.vendor_name AS focus_vendor_name
        , bi.vendor_name AS benchmark_vendor_name

        , fi.extended_item_amount_calc AS focus_amount
        , bi.extended_item_amount_calc AS benchmark_amount
        , e.extended_estimate_item_amount_calc AS estimate_amount

        , (fi.extended_item_amount_calc - bi.extended_item_amount_calc) AS focus_vs_benchmark_amount

        , CASE
            WHEN fi.extended_item_amount_calc > bi.extended_item_amount_calc THEN 'Overbid'
            WHEN fi.extended_item_amount_calc < bi.extended_item_amount_calc THEN 'Underbid'
            ELSE 'Tie'
          END AS over_under_flag

        , ABS(fi.extended_item_amount_calc - bi.extended_item_amount_calc) AS abs_diff

        , CASE
            WHEN bi.extended_item_amount_calc IS NOT NULL
             AND bi.extended_item_amount_calc <> 0
            THEN fi.extended_item_amount_calc / bi.extended_item_amount_calc
          END AS focus_vs_benchmark_ratio

        , (fi.extended_item_amount_calc - e.extended_estimate_item_amount_calc) AS focus_vs_estimate_amount

    FROM item_universe u
    LEFT JOIN focus_items fi
      ON u.project_id = fi.project_id
     AND u.bid_item_sequence_number = fi.bid_item_sequence_number
     AND u.item_specification_key = fi.item_specification_key

    LEFT JOIN benchmark_items bi
      ON u.project_id = bi.project_id
     AND u.bid_item_sequence_number = bi.bid_item_sequence_number
     AND u.item_specification_key = bi.item_specification_key

    LEFT JOIN estimate_side e
      ON u.project_id = e.project_id
     AND u.bid_item_sequence_number = e.bid_item_sequence_number
     AND u.item_specification_key = e.item_specification_key
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_TABLE}
    """).collect()[0]["row_count"]

    log_run(
        "SUCCESS",
        row_count,
        "Built gold.fact_project_item_comparison successfully."
    )

    print("=" * 70)
    print("Built gold.fact_project_item_comparison")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise